# Preparation carbon density data (figure m02 and m04)

In [ ]:
# Libraries
import pandas as pd
import dask.dataframe as dd

In [ ]:
# Directories
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'
dir08 = '../paper_ecosystem_carbon/output/08_eval/'
dir_out = '../paper_ecosystem_carbon/output/zenodo/'

---

In [ ]:
# Initialize dask
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(queue='interactive',
                       cores=24,
                       memory='100 GB',
                       account='bm0891', 
                       interface="ib0",
                       walltime='02:00:00',
                       local_directory='../dask/',
                       job_extra_directives=[
                           '-o ../dask/LOG_worker_%j.o',
                           '-e ../dask/LOG_worker_%j.o'])

# Scale dask clusters
cluster.scale(jobs=2)

# Configurate dashboard url
import dask
dask.config.config.get('distributed').get('dashboard') \
    .update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Create client
from dask.distributed import Client
client = Client(cluster)
client

---

### Get data

In [ ]:
# Get database prepared for random forest models
dd_base = dd.read_parquet(f'{dir02}dd_base_r001.parquet')

In [ ]:
# Get literature data
df_lit_cveg = pd.read_excel(
    f'{dir08}tables/literature_carbon_collection_20260818.xlsx',
    sheet_name='cveg_all')

df_lit_csoil = pd.read_excel(
    f'{dir08}tables/literature_carbon_collection_20260818.xlsx',
    sheet_name='csoil_all')

In [ ]:
# Ecosystem categories
dict_etype = dict(trop_ef = 'Tropical/subtropical evergreen forest', 
                  trop_df = 'Tropical/subtropical deciduous forest',
                  temp_ebf = 'Temperate/boreal evergreen broadleaf forest',
                  temp_dbf = 'Temperate/boreal deciduous broadleaf forest',
                  temp_enf = 'Temperate/boreal evergreen needleleaf forest',
                  temp_dnf = 'Temperate/boreal deciduous needleleaf forest',
                  glob_shrub = 'Shrublands',
                  glob_grass = 'Grasslands',
                  glob_tundr = 'Tundra',
                  glob_mount = 'Montane')

---

### Prepare training data csv

In [ ]:
def _get_train_data(ctype, etype):

    """Helper function: Get training data"""
    
    # Define cbase (ensure consistency with 02_05_prep_database)
    if ctype == 'cveg':
        if etype in ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
                     'temp_dnf', 'glob_shrub']:
            cbase = 'cveg_biom'    
        if etype in ['glob_tundr', 'glob_mount']:
            cbase = 'cveg_tund'
        if etype in ['glob_grass', 'glob_past']:
            cbase = 'cveg_gras'
        if etype == 'glob_crop':
            cbase = 'cveg_crop'
    if ctype == 'csoil':
        cbase = 'csoil_000_100'

    # Get training data values
    x = dd_base_sel[dd_base_sel[f'train_{ctype}_{etype}'] == True][cbase].values.compute()
    return pd.Series(x)
    

# Prepare dd_base
dd_base_sel = dd_base[
    ['cveg_biom', 'cveg_tund', 'cveg_gras', 'csoil_000_100',
    *[i for i in dd_base.columns if i.startswith('train_')]]] \
    .repartition(partition_size='200 MiB') \
    .persist()

# Create dataframe with training data
dd_train = pd.DataFrame()

for ctype in ['cveg', 'csoil']:
    for etype in dict_etype.keys():
        
        data = _get_train_data(ctype, etype).rename(f'{ctype}_{etype}')
        dd_train = pd.concat([dd_train, data], axis=1)

# Convert to integer
dd_train = dd_train.round(0).astype('Int16')

# Check
assert set(dd_train.columns) == {
    f'{ctype}_{etype}'
    for ctype in ['cveg', 'csoil']
    for etype in dict_etype
}

In [ ]:
# Comment and export
file_path = f'{dir_out}ecosystem_carbon_data_m02_m04_training.csv'

with open(file_path, 'w', encoding='utf-8') as f:

    f.write('metadata,ecosystem_carbon_data_m02_m04_training\n')
    f.write('variables,cveg = vegetation carbon density\n')
    f.write('variables,csoil = soil organic carbon density (0-1 m)\n')
    f.write('units,tC ha-1\n')

    for etype, description in dict_etype.items():
        f.write(f'ecosystem_type,{etype} = {description}\n')

    f.write(
        'data_structure,Each column contains an independent training sample\n'
    )
    f.write(
        'data_structure,Rows are not matched across columns\n'
    )
    f.write(
    'data_structure,Empty cells are caused by different sample sizes\n'
)
    f.write('\n')
    
dd_train.to_csv(file_path, mode='a', index=False)

---

### Prepare literature data csv

In [ ]:
# Prepare vegetation carbon
df_cveg = df_lit_cveg.rename(
    columns={
        'agbc_bgbc_best': 'value_best',
        'agbc_bgbc_min': 'value_min',
        'agbc_bgbc_max': 'value_max',
    }
).copy()

df_cveg['carbon_type'] = 'cveg'

# Prepare soil carbon
df_csoil = df_lit_csoil.rename(
    columns={
        'soc_best': 'value_best',
        'soc_min': 'value_min',
        'soc_max': 'value_max',
    }
).copy()

df_csoil['carbon_type'] = 'csoil'

# Combine
df_lit = pd.concat([df_cveg, df_csoil], ignore_index=True)

# Order columns
df_lit = df_lit[['publication', 'assigned_eco_zone', 'assigned_name',
                 'carbon_type', 'value_best', 'value_min', 'value_max',
                 'comment',]]

# Remove empty rows
df_lit = df_lit.dropna(
    subset=['publication', 'assigned_eco_zone', 'assigned_name'],
    how='all'
).reset_index(drop=True)

# Bounds check
has_bounds = df_lit[['value_min', 'value_max']].notna().all(axis=1)

bad_bounds = df_lit.loc[
    has_bounds
    & ~(
        (df_lit['value_min'] <= df_lit['value_best'])
        & (df_lit['value_best'] <= df_lit['value_max'])
    )
]

print(bad_bounds)

# Check negative values
print(
    df_lit.loc[
        df_lit['value_min'] < 0,
        ['publication', 'assigned_name', 'carbon_type',
         'value_best', 'value_min', 'value_max']
    ]
)

# Check
assert set(df_lit['carbon_type']) == {'cveg', 'csoil'}
assert df_lit['publication'].notna().all()
assert df_lit['assigned_name'].notna().all()
assert df_lit['value_best'].notna().all()

In [ ]:
# Comment and export
file_path = f'{dir_out}ecosystem_carbon_data_m02_m04_literature.csv'

with open(file_path, 'w', encoding='utf-8') as f:

    f.write('metadata,ecosystem_carbon_data_m02_m04_literature\n')
    f.write('variables,cveg = vegetation carbon density\n')
    f.write('variables,csoil = soil organic carbon density (0-1 m)\n')
    f.write('units,tC ha-1\n')
    f.write('\n')
    
df_lit.to_csv(file_path, mode='a', index=False)

---

### Check

In [ ]:
file_path_train = f'{dir_out}ecosystem_carbon_data_m02_m04_training.csv'

pd_train_check = pd.read_csv(file_path_train, skiprows=18)
print(pd_train_check)

In [ ]:
file_path_lit = f'{dir_out}ecosystem_carbon_data_m02_m04_literature.csv'
pd_lit_check = pd.read_csv(file_path_lit, skiprows=5)
print(pd_lit_check)